In [ ]:
"""
Toy variational inference with normalizing flows, matching the three notebook pages:

Page 1: affine coupling layer
Page 2: change-of-variables density q_lambda(y)
Page 3: reverse-KL variational objective

The script also includes planar flow for comparison.

Target density (unnormalized posterior):
    g(theta) = p(theta) p(D | theta)
where theta = (theta1, theta2),
      p(theta) = N(0, I),
      x_obs | theta ~ N(theta1 + theta2^2, sigma_obs^2).

This produces a non-Gaussian / banana-like posterior that is a useful toy target
for comparing affine coupling flow and planar flow.

Usage examples:
    python affine_coupling_vi_demo.py --flow both --steps 4000 --plot
    python affine_coupling_vi_demo.py --flow affine --steps 3000
"""

In [1]:
from __future__ import annotations

import argparse
import math
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F  # using softplus


def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# Target posterior: log g(y)
def standard_normal_logprob(x):
    dim = x.shape[-1]
    return -0.5 * (x.pow(2).sum(dim=-1) + dim * math.log(2.0 * math.pi))


def banana_log_g(
    theta: torch.Tensor,
    x_obs: float = 1.0,
    sigma_obs: float = 0.15,
):
    """
    Unnormalized log target density:
        log g(theta) = log p(theta) + log p(x_obs | theta)

    Model:
        theta ~ N(0, I)
        x_obs | theta ~ N(theta1 + theta2^2, sigma_obs^2)
    """
    theta1 = theta[..., 0]
    theta2 = theta[..., 1]

    log_prior = standard_normal_logprob(theta)
    mean = theta1 + theta2.pow(2)
    resid = x_obs - mean
    log_lik = -0.5 * ((resid / sigma_obs) ** 2 + math.log(2.0 * math.pi * sigma_obs**2))
    return log_prior + log_lik


# Small helper network for s,t
class MLP(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


In [15]:
# affine coupling layer
class AffineCoupling(nn.Module):
    """
    Vector version of Real NVP affine coupling:

        y = b ⊙ x + (1-b) ⊙ [ x ⊙ exp(s(b ⊙ x)) + t(b ⊙ x) ]

    where b is a binary mask.

    Notes:
    - log|det J| only depends on the scale outputs s on the transformed block.
    - inverse is closed-form.
    """

    def __init__(self, dim: int, mask: torch.Tensor, hidden_dim: int = 64) -> None:
        super().__init__()
        if mask.shape != (dim,):
            raise ValueError(f"mask must have shape ({dim},), got {tuple(mask.shape)}")
        self.dim = dim
        self.register_buffer("mask", mask.float())
        self.scale_net = MLP(dim, dim, hidden_dim)
        self.shift_net = MLP(dim, dim, hidden_dim)
        # Stabilize the scale; this is analogous to using tanh in Real NVP.
        self.scale_factor = nn.Parameter(torch.tensor(0.5))

    def _scale_and_shift(self, x_masked: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        raw_scale = self.scale_net(x_masked)
        scale = torch.tanh(raw_scale) * self.scale_factor.exp()
        shift = self.shift_net(x_masked)
        # Only the unmasked block is actually transformed.
        scale = scale * (1.0 - self.mask)
        shift = shift * (1.0 - self.mask)
        return scale, shift

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x_masked = x * self.mask
        scale, shift = self._scale_and_shift(x_masked)
        y = x_masked + (1.0 - self.mask) * (x * torch.exp(scale) + shift)
        logabsdet = scale.sum(dim=-1)
        return y, logabsdet

    def inverse(self, y: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        y_masked = y * self.mask
        scale, shift = self._scale_and_shift(y_masked)
        x = y_masked + (1.0 - self.mask) * ((y - shift) * torch.exp(-scale))
        logabsdet = (-scale).sum(dim=-1)
        return x, logabsdet


class AffineCouplingFlow(nn.Module):
    def __init__(self, dim: int, num_layers: int = 8, hidden_dim: int = 64):
        super().__init__()
        if dim < 2:
            raise ValueError("Affine coupling flow needs dim >= 2")

        layers: List[AffineCoupling] = []
        for k in range(num_layers):
            if dim == 2:
                mask = torch.tensor([1.0, 0.0]) if k % 2 == 0 else torch.tensor([0.0, 1.0])
            else:
                mask = torch.tensor([(i + k) % 2 for i in range(dim)], dtype=torch.float32)
            layers.append(AffineCoupling(dim, mask, hidden_dim))
        self.layers = nn.ModuleList(layers)
        self.dim = dim

    def forward(self, u: torch.Tensor):
        z = u
        total_logabsdet = torch.zeros(u.shape[0], device=u.device)
        for layer in self.layers:
            z, logabsdet = layer(z)
            total_logabsdet = total_logabsdet + logabsdet
        return z, total_logabsdet

    def inverse(self, y: torch.Tensor):
        z = y
        total_logabsdet = torch.zeros(y.shape[0], device=y.device)
        for layer in reversed(self.layers):
            z, logabsdet = layer.inverse(z)
            total_logabsdet = total_logabsdet + logabsdet
        return z, total_logabsdet

    def sample(self, n: int, device: Optional[torch.device] = None):
        device = device or next(self.parameters()).device
        u = torch.randn(n, self.dim, device=device)
        y, _ = self.forward(u)
        return y

    def log_prob(self, y: torch.Tensor): # q_lambda(y) = p0(H(y)) * |det J_H(y)|
      
        u, inv_logabsdet = self.inverse(y)
        return standard_normal_logprob(u) + inv_logabsdet

In [22]:
# Planar flow for comparison
class PlanarFlow(nn.Module):
    """
    f(z) = z + u h(w^T z + b)

    This is the planar flow from Rezende & Mohamed (2015).
    We implement the standard invertibility-friendly reparameterization of u.

    Important contrast with affine coupling:
    - forward and log-det are easy;
    - inverse is not available in general in closed form.
    """

    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        self.u = nn.Parameter(torch.randn(dim) * 0.01)
        self.w = nn.Parameter(torch.randn(dim) * 0.01)
        self.b = nn.Parameter(torch.zeros(()))

    @staticmethod
    def _m(x: torch.Tensor):
        return -1.0 + F.softplus(x)

    def _u_hat(self) -> torch.Tensor:
        wu = (self.w * self.u).sum()
        w_norm_sq = self.w.sum().pow(2) + 1e-8
        return self.u + ((self._m(wu) - wu) * self.w / w_norm_sq)

    def forward(self, z: torch.Tensor):
        u_hat = self._u_hat()
        linear = z @ self.w + self.b
        h = torch.tanh(linear)
        z_new = z + h.unsqueeze(-1) * u_hat.unsqueeze(0)

        # psi(z) = h'(w^T z + b) * w
        hprime = 1.0 - torch.tanh(linear).pow(2)
        psi = hprime.unsqueeze(-1) * self.w.unsqueeze(0)
        det_term = 1.0 + psi @ u_hat
        logabsdet = torch.log(det_term.abs() + 1e-8)
        return z_new, logabsdet


class PlanarFlowModel(nn.Module):
    def __init__(self, dim: int, num_layers: int = 16):
        super().__init__()
        self.layers = nn.ModuleList([PlanarFlow(dim) for _ in range(num_layers)])
        self.dim = dim

    def forward(self, u: torch.Tensor):
        z = u
        total_logabsdet = torch.zeros(u.shape[0], device=u.device)
        for layer in self.layers:
            z, logabsdet = layer(z)
            total_logabsdet = total_logabsdet + logabsdet
        return z, total_logabsdet

    def sample(self, n: int, device: Optional[torch.device] = None):
        device = device or next(self.parameters()).device
        u = torch.randn(n, self.dim, device=device)
        y, _ = self.forward(u)
        return y

In [17]:
# reverse-KL objective
def reverse_kl_loss(flow: nn.Module, batch_size: int, device: torch.device):
    """
    Monte Carlo estimator of the notebook's objective (up to the unknown constant log C):

        L(lambda) = E_{u ~ p0}
            [ log p0(u) - log |det J_T(u)| - log g(T(u)) ] + const.

    Here p0 is fixed to a standard Gaussian, so there is no separate gradient term for
    learnable base parameters.
    """
    u = torch.randn(batch_size, 2, device=device)
    y, logabsdet = flow(u)
    log_p0 = standard_normal_logprob(u)
    log_g = banana_log_g(y)
    return (log_p0 - logabsdet - log_g).mean()


@torch.no_grad()
def estimate_objective(flow: nn.Module, num_samples: int, device: torch.device):
    u = torch.randn(num_samples, 2, device=device)
    y, logabsdet = flow(u)
    value = (standard_normal_logprob(u) - logabsdet - banana_log_g(y)).mean()
    return float(value.cpu())


@dataclass
class TrainResult:
    name: str
    model: nn.Module
    losses: List[float]
    final_estimate: float


def train_flow(
    model: nn.Module,
    name: str,
    steps: int,
    lr: float,
    batch_size: int,
    device: torch.device,
    print_every: int = 500,
) -> TrainResult:
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses: List[float] = []

    for step in range(1, steps + 1):
        optimizer.zero_grad(set_to_none=True)
        loss = reverse_kl_loss(model, batch_size=batch_size, device=device)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach().cpu()))

        if step % print_every == 0 or step == 1 or step == steps:
            print(f"[{name}] step={step:5d}  loss={loss.item():.4f}")

    final_estimate = estimate_objective(model, num_samples=20_000, device=device)
    print(f"[{name}] estimated objective after training: {final_estimate:.4f}")
    return TrainResult(name=name, model=model, losses=losses, final_estimate=final_estimate)


In [18]:
def maybe_plot(results: Sequence[TrainResult], outdir: Path, device: torch.device) -> None:
    try:
        import matplotlib.pyplot as plt
    except Exception as exc:
        print(f"matplotlib is unavailable, skip plotting: {exc}")
        return

    outdir.mkdir(parents=True, exist_ok=True)

    xs = torch.linspace(-3.0, 3.0, 220)
    ys = torch.linspace(-3.0, 3.0, 220)
    X, Y = torch.meshgrid(xs, ys, indexing="xy")
    grid = torch.stack([X.reshape(-1), Y.reshape(-1)], dim=-1).to(device)
    log_g = banana_log_g(grid).reshape(X.shape).cpu()

    fig1 = plt.figure(figsize=(6, 4))
    for result in results:
        plt.plot(result.losses, label=result.name)
    plt.xlabel("training step")
    plt.ylabel("MC estimate of reverse-KL objective")
    plt.legend()
    plt.tight_layout()
    path1 = outdir / "training_losses.png"
    fig1.savefig(path1, dpi=160)
    plt.close(fig1)
    print(f"saved {path1}")

    ncols = len(results) + 1
    fig2, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4), squeeze=False)
    axes = axes[0]

    axes[0].contourf(X.numpy(), Y.numpy(), torch.exp(log_g).numpy(), levels=40)
    axes[0].set_title("Target density exp(log g)")
    axes[0].set_xlim(-3, 3)
    axes[0].set_ylim(-3, 3)

    for ax, result in zip(axes[1:], results):
        samples = result.model.sample(6000, device=device).cpu()
        ax.contour(X.numpy(), Y.numpy(), log_g.numpy(), levels=18, linewidths=0.8)
        ax.scatter(samples[:, 0].detach().numpy(), samples[:, 1].detach().numpy(), s=3, alpha=0.25)
        ax.set_title(f"{result.name}\nobjective={result.final_estimate:.3f}")
        ax.set_xlim(-3, 3)
        ax.set_ylim(-3, 3)

    fig2.tight_layout()
    path2 = outdir / "flow_samples_vs_target.png"
    fig2.savefig(path2, dpi=160)
    plt.close(fig2)
    print(f"saved {path2}")

    # Demonstrate page-2 density evaluation for affine coupling only.
    for result in results:
        if isinstance(result.model, AffineCouplingFlow):
            y0 = torch.tensor([[0.5, 0.5]], device=device)
            log_q = result.model.log_prob(y0).item()
            text_path = outdir / "affine_logprob_demo.txt"
            text_path.write_text(
                "At y=(0.5, 0.5), affine coupling flow can evaluate log q(y) via inverse:\n"
                f"log q(y) = {log_q:.6f}\n"
                "This matches page-2 of the notebook: q_lambda(y) = p0(H(y)) |det J_H(y)|.\n",
                encoding="utf-8",
            )
            print(f"saved {text_path}")
            break



def parse_args(args=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--flow", choices=["affine", "planar", "both"], default="both")
    parser.add_argument("--steps", type=int, default=4000)
    parser.add_argument("--batch-size", type=int, default=512)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--seed", type=int, default=7)
    parser.add_argument("--device", type=str, default="cpu")
    parser.add_argument("--plot", action="store_true")
    parser.add_argument("--outdir", type=str, default="./flow_outputs")
    parser.add_argument("--affine-layers", type=int, default=8)
    parser.add_argument("--planar-layers", type=int, default=16)
    parser.add_argument("--hidden-dim", type=int, default=64)
    return parser.parse_args(args=args)


def main(args=None):
    args = parse_args(args)
    set_seed(args.seed)

    device = torch.device(args.device)
    outdir = Path(args.outdir)

    results: List[TrainResult] = []

    if args.flow in ("affine", "both"):
        affine = AffineCouplingFlow(dim=2, num_layers=args.affine_layers, hidden_dim=args.hidden_dim)
        result = train_flow(
            model=affine,
            name="affine-coupling",
            steps=args.steps,
            lr=args.lr,
            batch_size=args.batch_size,
            device=device,
            print_every=max(1, args.steps // 8),
        )
        results.append(result)

    if args.flow in ("planar", "both"):
        planar = PlanarFlowModel(dim=2, num_layers=args.planar_layers)
        result = train_flow(
            model=planar,
            name="planar",
            steps=args.steps,
            lr=args.lr,
            batch_size=args.batch_size,
            device=device,
            print_every=max(1, args.steps // 8),
        )
        results.append(result)

    print("\nSummary:")
    for result in results:
        print(f"  {result.name:16s} objective ≈ {result.final_estimate:.4f}")

    if args.plot:
        maybe_plot(results, outdir=outdir, device=device)

In [23]:
main([
    "--flow", "both",
    "--steps", "2000",
    "--plot",
    "--outdir", "./flow_outputs"
])

[affine-coupling] step=    1  loss=4749.0435
[affine-coupling] step=  250  loss=2.1465
[affine-coupling] step=  500  loss=1.8977
[affine-coupling] step=  750  loss=1.6545
[affine-coupling] step= 1000  loss=1.5922
[affine-coupling] step= 1250  loss=1.4398
[affine-coupling] step= 1500  loss=1.3535
[affine-coupling] step= 1750  loss=1.4059
[affine-coupling] step= 2000  loss=1.3207
[affine-coupling] estimated objective after training: 1.3425
[planar] step=    1  loss=29.5742
[planar] step=  250  loss=26.6242
[planar] step=  500  loss=20.6759
[planar] step=  750  loss=11.4902
[planar] step= 1000  loss=5.1706
[planar] step= 1250  loss=4.3972
[planar] step= 1500  loss=4.4199
[planar] step= 1750  loss=4.3364
[planar] step= 2000  loss=4.3547
[planar] estimated objective after training: 4.3517

Summary:
  affine-coupling  objective ≈ 1.3425
  planar           objective ≈ 4.3517
saved flow_outputs\training_losses.png
saved flow_outputs\flow_samples_vs_target.png
saved flow_outputs\affine_logprob_